## 1. Auswertung & Diskussion: Vergleich der Benchmarks (Aufgabe b & c)

<div style="display: flex; justify-content: space-around;">
<img src="./images/benchmark_results/bench2dof.png" width="40%">
<img src="./images/benchmark_results/bench3dof.png" width="40%">
</div>

<div align="center">
<div style="width: 40%;">
<img src="./images/benchmark_results/bench3dofPlanar.png" style="width: 100%;">
<p><i>Abbildung 3: Planararm Benchmarks</i></p>
</div>
</div>

<div style="display: flex; justify-content: space-around;">
<img src="./images/benchmark_results/bench2dofBalken.png" width="40%">
<img src="./images/benchmark_results/bench3dofBalken.png" width="40%">
</div>

<div align="center">
<div style="width: 40%;">
<img src="./images/benchmark_results/bench3dofPlanarBalken.png" style="width: 100%;">
<p><i>Planararm Balkendiagramm</i></p>
</div>
</div>

### Analyse der Ergebnisse

**Planungszeit:**
Der **LazyPRM** bestätigt sich über alle Szenarien hinweg als der effizienteste Planer für Einzelanfragen. Im komplexen Szenario *PlanarArm_2DoF* erzielt er eine Zeit von **2,378 s**, während der **BasicPRM** mit **269,784 s** massiv abfällt. Selbst bei den 3-DoF ShapeRobotern (spinner/grid) bleibt LazyPRM mit Zeiten um **1,1 s** extrem performant.

* **Grund:** Die Strategie, Kollisionsprüfungen zu verzögern, reduziert die `CollChecks` dramatisch (z. B. nur **1.453** beim LazyPRM vs. **136.247** beim BasicPRM im PlanarArm_2DoF).
* Der **BasicPRM** weist konsequent die höchsten Laufzeiten auf, da er die gesamte Roadmap vorab validiert, was besonders im *PlanarArm_3DoF* zu einer Zeit von **730,877 s** führt.

**Roadmap-Topologie (Speichereffizienz):**
Der **Visibility-PRM (VisPRM)** demonstriert seine Stärke in der Erzeugung minimaler Roadmaps. Während Basic- und LazyPRM oft die maximalen Kapazitäten (bis zu 800 Knoten) nutzen, kommt der VisPRM mit einem Bruchteil davon aus.

* **Beispiel:** Nur **10 Knoten** beim `PlanarArm_2DoF` und lediglich **12 bis 14 Knoten** bei den 3-DoF Szenarien.
* **Erkenntnis:** VisPRM platziert Knoten nur dort, wo sie die Sichtbarkeit im Konfigurationsraum tatsächlich erweitern. Dies resultiert in einer extrem schlanken Datenstruktur, benötigt jedoch beim Aufbau durch  mehr Zeit als der Lazy-Ansatz.

---

## 2. Evaluation des MultiQuery-Konzepts (Aufgabe c)

### Vorteile (Laufzeitgewinn bei Komplexität)

Die Entwicklung des **MultiQuery-angepassten VISPRM** zeigt signifikante Performance-Vorteile in komplexen Umgebungen durch die einmalige Erstellung und anschließende Wiederverwendung der Visibility-Roadmap für den Roundtrip:

* **Massiver Speedup:** Bei den 3-DoF ShapeRobotern (spinner_3DoF) reduziert sich die Zeit von **1,364 s** (Standard VisPRM) auf beachtliche **0,219 s**.
* **Planar-Roboter:** Im rechenintensiven *PlanarArm_3DoF* sinkt die Planungszeit von **441,447 s** auf **146,413 s**, womit die Variante hier sogar schneller als der LazyPRM ist.
* **Kollisionsersparnis:** Die Anzahl der `CollChecks` sinkt beim MultiQuery-Ansatz im *PlanarArm_2DoF* von **33.574** auf **10.672**.

### Nachteile & Anomalien (Pfadqualität)

Trotz der verbesserten Laufzeit bleibt die Pfadqualität (Distanz) ein kritischer Faktor, der durch die spärliche Roadmap-Struktur beeinflusst wird:

* **Distanz-Penalty bei ShapeRobotern:** Beim `spinner_3DoF` ist der Pfad mit **1524,57 m** deutlich länger als beim Standard-VisPRM (**342,11 m**). Die Reduktion von  hat die Extremwerte (vorher über 4000m) verbessert, aber das Grundproblem der spärlichen Vernetzung bleibt bestehen.
* **Positive Anomalie:** Überraschenderweise liefert der MultiQuery-VisPRM beim *PlanarArm_3DoF* mit **39,69** eine bessere Distanz als der Standard-VisPRM mit **55,81**. Dies deutet darauf hin, dass eine globale Visibility-Roadmap für komplexe Manipulatoren vorteilhafter sein kann als mehrere lokale Roadmaps.

---

## 3. Zusammenfassendes Fazit

Die Evaluation an den 6 Benchmark-Umgebungen führt zu folgenden Kern-Erkenntnissen:

1. **Effizienz- und Qualitäts-Sieger:** Der **LazyPRM** ist der überlegene Allrounder. Er liefert nicht nur die **kürzesten Pfaddistanzen** in fast allen Szenarien (z. B. **22,56** beim *PlanarArm_2DoF* oder **165,66 m** bei *arcade_2DoF*), sondern ist dabei auch der schnellste Planer für Einzelabfragen.
2. **Struktur-Spezialist:** Der **VisPRM** optimiert konsequent die Roadmap-Größe und benötigt die wenigsten Knoten zur Problemlösung (oft nur 10-14 Knoten). Er ist ideal für Anwendungen mit begrenztem Speicherplatz, erzielt jedoch selten die kürzeste Pfadlänge, da die spärliche Vernetzung direkte Wege oft verhindert.
3. **MultiQuery-Strategie:** Die Entwicklung des **MultiQuery-angepassten VISPRM** konnte die Rechenzeit bei komplexen Roundtrips massiv senken (Speedup von über **80 %** beim *spinner_3DoF*).
* **Der Distanz-Trade-off:** Es wurde deutlich, dass die Pfadqualität bei diesem Ansatz am stärksten leidet. Da die einmalig erstellte, sehr dünne Visibility-Roadmap für alle Etappen des Roundtrips genutzt wird, entstehen oft deutliche Umwege im Vergleich zum flexibleren LazyPRM.


**Abschlussbewertung:**
Für Anwendungen, bei denen die kürzeste Distanz und schnelle Planung entscheidend sind, ist der **LazyPRM** die erste Wahl. Der **MultiQuery-angepasste VISPRM** ist eine hochperformante Lösung für rechenintensive Multi-Query-Aufgaben, sollte jedoch mit einem Glättungs-Algorithmus (Path Smoothing) kombiniert werden, um die Distanznachteile der spärlichen Roadmap auszugleichen.